## Start

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import datetime
import plotly.graph_objects as go


load_dotenv()
ticker = ''
path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)
path_analysis_csv = os.path.join(path_stockdata, f'{ticker}--Analysis_Ownership-s1v1.csv')

today = datetime.date.today()
cutoff = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3


## Data Imports & Cleaning

In [ ]:
%%capture
df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Annual_10K-s1v1.csv'), index_col=0)
df0 = df0.dropna(subset=['FiscalYear'])
df0[['FiscalYear', 'FiscalMonth']] = df0[['FiscalYear', 'FiscalMonth']].astype(int)

In [ ]:
df0

## Owner Return

In [ ]:
%%capture

owner_df = df0[['FiscalYear', 'Revenue', 'OpCash', 'FreeCash','DivCash', 'StockIssue', 'StockBuyBack',
                'C&E', 'TreasuryStock', 'SharesOutstandingBasic', 'SharesOutstandingDiluted']]
owner_df['OwnerStock'] = owner_df['StockIssue'] + owner_df['StockBuyBack']
owner_df['OwnerDiv'] = owner_df['DivCash']
owner_df['OwnerTot'] = owner_df['OwnerStock'] + owner_df['OwnerDiv']
owner_df['OwnerYield'] = owner_df['OwnerTot'] / owner_df['Revenue']

In [ ]:
owner_df.tail(10)

## Total Owner Return

In [ ]:
owner_return10 = owner_df['OwnerTot'].tail(10).sum()
owner_div10 = owner_df['OwnerDiv'].tail(10).sum()
owner_stock10 = owner_df['OwnerStock'].tail(10).sum()
rev10 = owner_df['Revenue'].tail(10).sum()
oc10 = owner_df['OpCash'].tail(10).sum()
fc10 = owner_df['FreeCash'].tail(10).sum()

print(f'Year10 Total Cast Return: $ {abs(owner_return10)}')
print(f'Year10 Dividend Return: $ {abs(owner_div10)}')
print(f'Year10 Stock B/I Return: $ {abs(owner_stock10)}')
print()
print(f'Percent of revenue returned to owners: {abs(round(owner_return10 / rev10 * 100, 2))} %')
print(f'Percent of operating cash returned to owners: {abs(round(owner_return10 / oc10 * 100, 2))} %')
print(f'Percent of free cash returned to owners: {abs(round(owner_return10 / fc10 * 100, 2))} %')

In [ ]:
owner_fig0 = go.Figure(data=[
    go.Bar(name='OpCash', x=owner_df['FiscalYear'].tail(10), y=owner_df['OpCash'].tail(10), offsetgroup=1, marker_color='blue'),
    go.Bar(name='FreeCash', x=owner_df['FiscalYear'].tail(10), y=owner_df['FreeCash'].tail(10), offsetgroup=2, marker_color='green'),
    go.Bar(name='OwnerDiv', x=owner_df['FiscalYear'].tail(10), y=owner_df['OwnerDiv'].tail(10).abs(), offsetgroup=3, marker_color='orange'),
    go.Bar(name='OwnerStock', x=owner_df['FiscalYear'].tail(10), y=owner_df['OwnerStock'].tail(10).abs(), offsetgroup=3,
           base=owner_df['OwnerDiv'].tail(10).abs(), marker_color='yellow')
])
owner_fig0.update_layout(barmode='group')
owner_fig0.update_xaxes(dtick=1)
owner_fig0.show()

In [ ]:
owner_fig1 = go.Figure(data=[
    go.Bar(name='OwnerYield', x=owner_df['FiscalYear'].tail(10), y=owner_df['OwnerYield'].tail(10).abs(), offsetgroup=1, marker_color='blue'),
])
owner_fig1.update_layout(barmode='group')
owner_fig1.update_layout(yaxis_title='Yield %', xaxis_title='FiscalYear', title='10 Year Owner Yield', template='plotly_dark')
owner_fig1.update_xaxes(dtick=1)
owner_fig1.show()

## Shares Out

In [ ]:
share_fig1 = go.Figure(data=[
    go.Bar(name='BasicShares', x=owner_df['FiscalYear'].tail(10), y=owner_df['SharesOutstandingBasic'].tail(10), offsetgroup=1, marker_color='blue'),
    go.Bar(name='DilutedShares', x=owner_df['FiscalYear'].tail(10), y=owner_df['SharesOutstandingDiluted'].tail(10), offsetgroup=2, marker_color='green')
])
share_fig1.update_layout(barmode='group')
share_fig1.update_layout(yaxis_title='Count', xaxis_title='FiscalYear', title='10 Year Share Count', template='plotly_dark')
share_fig1.update_xaxes(dtick=1)
share_fig1.show()

In [ ]:
%%capture

share_df = owner_df[['FiscalYear', 'SharesOutstandingDiluted', 'SharesOutstandingBasic']].tail(10)
share_df['DilutedChg'] = share_df['SharesOutstandingDiluted'].pct_change()
share_df['BasicChg'] = share_df['SharesOutstandingBasic'].pct_change()

share_df

In [ ]:
share_fig2 = go.Figure(data=[
   go.Bar(name='BasicChg', x=share_df['FiscalYear'], y=share_df['BasicChg'] * 100, offsetgroup=1, marker_color='blue'),
    go.Bar(name='DilutedChg', x=share_df['FiscalYear'], y=share_df['DilutedChg'] * 100, offsetgroup=2, marker_color='green')
])
share_fig2.update_xaxes(dtick=1)
share_fig2.update_layout(yaxis_title='% Change', xaxis_title='FiscalYear', title='10 Year Share Change', template='plotly_dark')
share_fig2.show()

## Dividend Returns

In [ ]:
%%capture

div_df = owner_df[['FiscalYear', 'Revenue', 'OpCash', 'FreeCash', 'DivCash']]
div_df['DivGro'] = div_df['DivCash'].pct_change()
div_df['DivMargin'] = div_df['DivCash'] / div_df['Revenue']
div_df['OpCashCover'] = div_df['DivCash'] / div_df['OpCash']
div_df['FreeCashCover'] = div_df['DivCash'] / div_df['FreeCash']


In [ ]:
div_df.tail(20)

In [ ]:
div_fig1 = go.Figure(data=[
   go.Bar(name='DivGro', x=div_df['FiscalYear'].tail(20), y=div_df['DivGro'].tail(20), offsetgroup=1, marker_color='blue')
    ])
div_fig1.update_xaxes(dtick=1)
div_fig1.update_layout(yaxis_title='% Change', xaxis_title='FiscalYear', title='10 Year DivGro', template='plotly_dark')
div_fig1.show()

In [ ]:
divgro_mean20 = div_df['DivGro'].tail(20).mean()
divgro_median20 = div_df['DivGro'].tail(20).median()

divgro_mean10 = div_df['DivGro'].tail(10).mean()
divgro_median10 = div_df['DivGro'].tail(10).median()

divgro_mean5 = div_df['DivGro'].tail(5).mean()
divgro_median5 = div_df['DivGro'].tail(5).median()

divgro_mean3 = div_df['DivGro'].tail(3).mean()
divgro_median3 = div_df['DivGro'].tail(3).median()

divgro_lst = div_df['DivGro'].iloc[-1]

print(f'Mean20Year DivGro: {round(divgro_mean20 * 100, 2)}%')
print(f'Median20Year DivGro: {round(divgro_median20 * 100, 2)}%')
print()
print(f'Mean10Year DivGro: {round(divgro_mean10 * 100, 2)}%')
print(f'Median10Year DivGro: {round(divgro_median10 * 100, 2)}%')
print()
print(f'Mean5Year DivGro: {round(divgro_mean5 * 100, 2)}%')
print(f'Median5Year DivGro: {round(divgro_median5 * 100, 2)}%')
print()
print(f'Mean3Year DivGro: {round(divgro_mean3 * 100, 2)}%')
print(f'Median3Year DivGro: {round(divgro_median3 * 100, 2)}%')
print()
print(f'Last DivGro: {round(divgro_lst * 100, 2)}%')

In [ ]:
div_fig2 = go.Figure(data=[
    go.Bar(name='OpCash', x=div_df['FiscalYear'].tail(20), y=round(div_df['OpCashCover'].tail(20).abs() * 100, 2), offsetgroup=1, marker_color='blue'),
    go.Bar(name='FreeCash', x=div_df['FiscalYear'].tail(20), y=round(div_df['FreeCashCover'].tail(20).abs() * 100, 2), offsetgroup=2, marker_color='Green')
    ])
div_fig2.update_xaxes(dtick=1)
div_fig2.update_yaxes(range=[0,100])
div_fig2.update_layout(yaxis_title='Coverage %', xaxis_title='FiscalYear', title='10 Year Div Coverage', template='plotly_dark')
div_fig2.show()

## Analysis Output

In [ ]:
metrics_json = {
    "analysis_type": "ownership",
    "analysis_date": today.strftime('%Y-%m-%d'),
    "last_df_date": df0['FiscalYear'].iloc[-1],
    "owner_cashsum10yr": abs(owner_return10),
    "owner_divsum10yr": abs(owner_div10),
    "owner_stocksum10yr": abs(owner_stock10),
    "owner_yield10yr": abs(round(owner_return10 / rev10, 4)),
    "mean20yr_divgro": divgro_median20,
    "median20yr_divgro": divgro_median20,
    "mean10yr_divgro": divgro_median10,
    "median10yr_divgro": divgro_median10,
    "mean5yr_divgro": divgro_median5,
    "median5yr_divgro": divgro_median5,
    "mean3yr_divgro": divgro_median3,
    "median3yr_divgro": divgro_median3,
    "divgro_trend": "slowing",
    "div_risk": "low"
}

metrics_json

In [ ]:
metrics_df = pd.DataFrame([metrics_json])
metrics_df
if os.path.isfile(path_analysis_csv):
    metrics_df.to_csv(path_analysis_csv, mode='a', header=False, index=False)
else:
    metrics_df.to_csv(path_analysis_csv, mode='w', header=True, index=False)

## End Notebook